<a href="https://colab.research.google.com/github/VidhiSidana/UCS420/blob/main/assignment4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
roll_number = input("Enter your roll number: ")
last_two_digits = roll_number[-2:]
fixed_entries = [
    {
        "question": "What is the annual fee?",
        "answer": "The annual fee is Rs 500.",
        "category": "billing",
        "keywords": ["fee", "cost", "price", "charge"]
    },
    {
        "question": "How do I reset my password?",
        "answer": "Go to Settings > Reset Password.",
        "category": "account",
        "keywords": ["password", "reset", "login"]
    },
    {
        "question": "What are your working hours?",
        "answer": "We are open 9 AM to 5 PM.",
        "category": "general",
        "keywords": ["hours", "timing", "open"]
    },
    {
        "question": "How can I pay the fee?",
        "answer": "You can pay via UPI, card, or net banking.",
        "category": "billing",
        "keywords": ["pay", "payment", "upi", "fee"]
    }
]
categories = ["billing", "account", "general"]
def create_personalized_entry(digit):
    category = categories[int(digit) % 3]

    if category == "billing":
        question = "How can I make a payment for my fees?"
        answer = "You can make the payment through UPI, card, or net banking."
        keywords = ["payment", "fees", "pay"]

    elif category == "account":
        question = "How do I update my registered account details?"
        answer = "You can update your account details from the account settings."
        keywords = ["account", "registered", "details"]

    else:
        question = "How can I get general information about the college?"
        answer = "You can contact the college help desk for general information."
        keywords = ["general", "information", "help"]

    return {
        "question": question,
        "answer": answer,
        "category": category,
        "keywords": keywords
    }
personalized_entry_1 = create_personalized_entry(last_two_digits[0])
personalized_entry_2 = create_personalized_entry(last_two_digits[1])
all_entries = fixed_entries + [
    personalized_entry_1,
    personalized_entry_2
]
df = pd.DataFrame(all_entries)
print("Final FAQ DataFrame:")
print(df)

Enter your roll number: 1024170410
Final FAQ DataFrame:
                                         question  \
0                         What is the annual fee?   
1                     How do I reset my password?   
2                    What are your working hours?   
3                          How can I pay the fee?   
4  How do I update my registered account details?   
5           How can I make a payment for my fees?   

                                              answer category  \
0                          The annual fee is Rs 500.  billing   
1                   Go to Settings > Reset Password.  account   
2                          We are open 9 AM to 5 PM.  general   
3         You can pay via UPI, card, or net banking.  billing   
4  You can update your account details from the a...  account   
5  You can make the payment through UPI, card, or...  billing   

                         keywords  
0      [fee, cost, price, charge]  
1        [password, reset, login]  
2       

In [2]:
def score_query(query, df):
    query_words = query.lower().split()

    results = []

    for index, row in df.iterrows():
        keywords = [word.lower() for word in row["keywords"]]

        score = 0

        for word in query_words:
            if word in keywords:
                score += 1

        if score > 0:
            results.append({
                "index": index,
                "question": row["question"],
                "answer": row["answer"],
                "category": row["category"],
                "score": score
            })
    results = sorted(results, key=lambda x: x["score"], reverse=True)

    return results

query = input("Enter your query: ")

results = score_query(query, df)

print("\nMatching FAQs:")

if results:
    for result in results:
        print(result)
else:
    print("No matching FAQ found.")

Enter your query: what is the annual fee

Matching FAQs:
{'index': 0, 'question': 'What is the annual fee?', 'answer': 'The annual fee is Rs 500.', 'category': 'billing', 'score': 1}
{'index': 3, 'question': 'How can I pay the fee?', 'answer': 'You can pay via UPI, card, or net banking.', 'category': 'billing', 'score': 1}


In [3]:
def same_category(category_name, df):
    result = df[df["category"].str.lower() == category_name.lower()]

    return result[["question", "answer", "category", "keywords"]]
category_to_search = personalized_entry_1["category"]

print("Category selected:", category_to_search)

print("\nFAQs belonging to this category:")
print(same_category(category_to_search, df))

Category selected: account

FAQs belonging to this category:
                                         question  \
1                     How do I reset my password?   
4  How do I update my registered account details?   

                                              answer category  \
1                   Go to Settings > Reset Password.  account   
4  You can update your account details from the a...  account   

                         keywords  
1        [password, reset, login]  
4  [account, registered, details]  


In [4]:
entry_index = 0
print("Selected FAQ:")
print(df.loc[entry_index])
new_keyword = input("\nEnter a new keyword: ")
df.at[entry_index, "keywords"].append(new_keyword)
print("\nUpdated FAQ:")
print(df.loc[entry_index])
filename = roll_number + "_faq_data.csv"

df.to_csv(filename, index=False)

print("\nData saved successfully as:", filename)

Selected FAQ:
question       What is the annual fee?
answer       The annual fee is Rs 500.
category                       billing
keywords    [fee, cost, price, charge]
Name: 0, dtype: object

Enter a new keyword: money

Updated FAQ:
question              What is the annual fee?
answer              The annual fee is Rs 500.
category                              billing
keywords    [fee, cost, price, charge, money]
Name: 0, dtype: object

Data saved successfully as: 1024170410_faq_data.csv


In [5]:
category_counts = df.groupby("category").size().reset_index(name="count")

print(category_counts)

  category  count
0  account      2
1  billing      3
2  general      1


In [6]:
def score_query_with_ties(query, df):
    query_words = query.lower().split()
    results = []
    for index, row in df.iterrows():

        keywords = [word.lower() for word in row["keywords"]]

        score = 0

        for word in query_words:
            if word in keywords:
                score += 1

        if score > 0:
            results.append({
                "index": index,
                "question": row["question"],
                "answer": row["answer"],
                "category": row["category"],
                "score": score
            })
    if not results:
        print("No matching FAQ found.")
        return
    highest_score = max(result["score"] for result in results)
    best_matches = [
        result for result in results
        if result["score"] == highest_score
    ]

    print("\nHighest Score:", highest_score)

    print("\nBest Matching FAQs:")

    for result in best_matches:
        print(result["question"])
        print(result["answer"])
        print(result["category"])
        print(result["score"])